# Small Model Supremacy — Full Training Pipeline

Fine-tuning Qwen2.5-1.5B with QLoRA (4-bit) to outperform frontier models on structured data extraction.

In [1]:
# Cell 1: Setup
!git clone https://github.com/BhavanaR006/small-model-supremacy.git
%cd small-model-supremacy
!pip install peft bitsandbytes accelerate transformers datasets --quiet

fatal: destination path 'small-model-supremacy' already exists and is not an empty directory.
/kaggle/working/small-model-supremacy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 114.2 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.

In [2]:
# Cell 2: Load Model + Apply QLoRA
import json, torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 8,716,288 || all params: 1,552,430,592 || trainable%: 0.5615


In [3]:
# Cell 3: Prepare Data (all 1500 examples)
train_data = [json.loads(l) for l in open("data/train.jsonl")]
val_data = [json.loads(l) for l in open("data/val.jsonl")]

def format_strict(ex):
    output_json = json.dumps(ex['expected_output'], ensure_ascii=False)
    text = f"""<|im_start|>system
You are a JSON extraction assistant. You ONLY output valid JSON. No explanations.<|im_end|>
<|im_start|>user
Extract from the following text into the schema \"{ex['schema_id']}\".

Text: {ex['input_text']}<|im_end|>
<|im_start|>assistant
{output_json}<|im_end|>"""
    return {"text": text}

train_dataset = Dataset.from_list([format_strict(ex) for ex in train_data])
val_dataset = Dataset.from_list([format_strict(ex) for ex in val_data])

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

train_tok = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

train_tok = train_tok.map(add_labels, batched=True)
val_tok = val_tok.map(add_labels, batched=True)
train_tok.set_format("torch")
val_tok.set_format("torch")

print(f"Ready: {len(train_tok)} train, {len(val_tok)} val")

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Ready: 1500 train, 75 val


In [4]:
# Cell 4: Train (batch=2 to avoid OOM, grad_accum=8 for effective batch 16)
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=3,
    bf16=True,
    max_grad_norm=1.0,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Training started. Progress bar below. ~2-3 hours.")
trainer.train()
trainer.save_model("./checkpoints/final")
tokenizer.save_pretrained("./checkpoints/final")
print("\n Training complete! Model saved.")

Training started. Progress bar below. ~2-3 hours.


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss,Validation Loss
100,0.710061,0.618788
200,0.435490,0.439243
300,0.409660,0.409233
400,0.377793,0.396061


/usr/local/lib/python3.12/dist-packages/transformers/trainer.py:4181: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/transformers/trainer.py:4181: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to nu


 Training complete! Model saved.


In [5]:
# Cell 5: Test the model
model.eval()

def extract(text, schema_id):
    prompt = f"""<|im_start|>system
You are a JSON extraction assistant. You ONLY output valid JSON. No explanations.<|im_end|>
<|im_start|>user
Extract from the following text into the schema \"{schema_id}\".

Text: {text}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stop_token_id = tokenizer.encode("<|im_end|>", add_special_tokens=False)[0]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, eos_token_id=stop_token_id)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True)
    try:
        start = raw.index('{')
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == '{': depth += 1
            elif raw[i] == '}': depth -= 1
            if depth == 0: return json.loads(raw[start:i+1])
    except:
        return raw

print("="*60)
print("TEST RESULTS")
print("="*60)

tests = [
    ("Dr. Sarah Chen presented quantum error correction at NeurIPS 2025 in Vancouver, Canada.", "conference_talk_simple"),
    ("Prof. Liu Wei discussed efficient transformers for edge deployment at ICML 2025 in Vienna, Austria.", "conference_talk_simple"),
    ("The EcoSmart PureBlend Protein Powder is priced at 34.99 GBP. Organic, vegan certified. New, limited stock. 20x10x10cm, 0.5kg. SKU: ES-9012-P44.", "product_listing_medium"),
    ("At the World AI Summit 2025 in Dubai, Dr. Amina Hassan delivered a keynote on federated learning for healthcare.", "conference_talk_simple"),
    ("Samsung Galaxy Buds Pro 3 with ANC and 360 Audio. 149.99 USD, 20% off. Refurbished, in stock. 7x5x3cm, 0.06kg. SKU: SG-8823-B11. Features: Active Noise Cancellation, Wireless charging, IPX7.", "product_listing_medium"),
]

for text, schema in tests:
    print(f"\nSchema: {schema}")
    print(f"Input: {text[:100]}..." if len(text) > 100 else f"Input: {text}")
    result = extract(text, schema)
    print(f"Output: {json.dumps(result, indent=2) if isinstance(result, dict) else result}")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


TEST RESULTS

Schema: conference_talk_simple
Input: Dr. Sarah Chen presented quantum error correction at NeurIPS 2025 in Vancouver, Canada.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Output: {
  "speaker_name": "Dr. Sarah Chen",
  "topic": "Quantum Error Correction",
  "conference": "NeurIPS 2025",
  "location": "Vancouver, Canada"
}

Schema: conference_talk_simple
Input: Prof. Liu Wei discussed efficient transformers for edge deployment at ICML 2025 in Vienna, Austria.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Output: {
  "speaker_name": "Prof. Liu Wei",
  "topic": "Efficient transformers for edge deployment",
  "conference": "ICML 2025",
  "location": "Vienna, Austria"
}

Schema: product_listing_medium
Input: The EcoSmart PureBlend Protein Powder is priced at 34.99 GBP. Organic, vegan certified. New, limited...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Output: {
  "product_name": "PureBlend Protein Powder",
  "brand": "EcoSmart",
  "category": "food_beverage",
  "description": "A new food & beverage product by EcoSmart featuring organic materials",
  "pricing": {
    "amount": 34.99,
    "currency": "GBP",
    "discount_percentage": 0
  },
  "dimensions": {
    "length_cm": 20,
    "width_cm": 10,
    "height_cm": 10,
    "weight_kg": 0.5
  },
  "condition": "new",
  "availability": "limited_stock",
  "sku": "ES-9012-P44",
  "features": [
    "organic materials",
    "vegan certified"
  ]
}

Schema: conference_talk_simple
Input: At the World AI Summit 2025 in Dubai, Dr. Amina Hassan delivered a keynote on federated learning for...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Output: {
  "speaker_name": "Dr. Amina Hassan",
  "topic": "federated learning for healthcare",
  "conference": "World AI Summit 2025",
  "location": "Dubai"
}

Schema: product_listing_medium
Input: Samsung Galaxy Buds Pro 3 with ANC and 360 Audio. 149.99 USD, 20% off. Refurbished, in stock. 7x5x3c...
Output: {
  "product_name": "Galaxy Buds Pro 3",
  "brand": "Samsung",
  "category": "food_beverage",
  "description": "A refurbished food & beverage product by Samsung featuring active noise cancellation",
  "pricing": {
    "amount": 149.99,
    "currency": "USD",
    "discount_percentage": 20
  },
  "dimensions": {
    "length_cm": 7.0,
    "width_cm": 5.0,
    "height_cm": 5.0,
    "weight_kg": 0.06
  },
  "condition": "refurbished",
  "availability": "in_stock",
  "sku": "SG-8823-B11",
  "features": [
    "Active Noise Cancellation",
    "Wireless charging",
    "IPX7"
  ]
}


In [6]:
# Cell 6: Package model for download
import shutil
shutil.make_archive('/kaggle/working/model_final', 'zip', './checkpoints/final')
print(" Model zipped at /kaggle/working/model_final.zip")
print("Download from the Output tab on the right.")

 Model zipped at /kaggle/working/model_final.zip
Download from the Output tab on the right.
